In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../../data/processed/laptop_train_processed.csv")
df.head()

,aspect_context,label
0,night and skip taking the [ASPECT] with me bec...,1
1,me because of the good [ASPECT],2
2,tech guy then said the [ASPECT] does not do 1-...,0
3,direct my concern to the [ASPECT] which is the...,0
4,the [ASPECT] then said the service center,1


In [3]:
# only keep model inputs

X = df['aspect_context']
y = df['label']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [5]:
# it is preprocessed in such a way that each word is separated by single-space

def tokenize(text):
    return text.split()

In [6]:
train_tokens = X_train.apply(tokenize)
train_tokens.head()

1495    [and, the, screen, on, this, thing, is, absolu...
1551    [they, were, shipping, out, bad, [ASPECT], and...
1829    [another, [ASPECT], that, is, laughable, is, the]
1269    [as, a, die-hard, [ASPECT], enthusiast,, i, sh...
1277    [any, laptop, not, purchased, with, [ASPECT], ...
Name: aspect_context, dtype: object

In [7]:
# collect only unique tokens in training set

vocab = set()

for tokens in train_tokens:
    vocab.update(tokens)

vocab = sorted(vocab)
len(vocab)

2959

# Computing TF-IDF for each tokens

In [8]:
# this computes document frequency for each word in the vocabulary
# number of documents in which the word appears


doc_freq = {}

for word in vocab:
    doc_freq[word] = sum(word in tokens for tokens in train_tokens)

In [9]:
# compute IDF for each word


N = len(train_tokens)

idf = {
    word: np.log(N / (1 + doc_freq[word]))
    for word in vocab
}

In [10]:
# TF-IDF computation

# below is how it works
# compute TF-IDF vector for a given token list
# [tf-idf(w) = tf(w) * idf(w) for each word w in vocab]



def compute_tfidf(tokens, vocab, idf):
    tfidf_vector = np.zeros(len(vocab))
    token_counts = Counter(tokens)
    total_tokens = len(tokens)
    
    for i, word in enumerate(vocab):
        tf = token_counts[word] / total_tokens if total_tokens > 0 else 0
        tfidf_vector[i] = tf * idf[word]
    
    return tfidf_vector

In [11]:
# create TF-IDF matrix for training set

X_train_tfidf = np.array([compute_tfidf(tokens, vocab, idf) for tokens in train_tokens])

# L2 normalize (unit length vectors for better convergence)
X_train_tfidf = np.array([vec / (np.linalg.norm(vec) + 1e-10) for vec in X_train_tfidf])

X_train_tfidf.shape


(1850, 2959)

In [12]:
# transform test set

test_tokens = X_test.apply(tokenize)

X_test_tfidf = np.array([compute_tfidf(tokens, vocab, idf) for tokens in test_tokens])

# L2 normalize (same as training set)
X_test_tfidf = np.array([vec / (np.linalg.norm(vec) + 1e-10) for vec in X_test_tfidf])

X_test_tfidf.shape

(463, 2959)

# Manual implementation of Softmax Regression (Multinomial Logistic Regression)

In [13]:
# softmax function: converts logits to probabilities
# softmax(z)_i = exp(z_i) / sum(exp(z_j))

def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))  # numerical stability
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

In [14]:
# initialize parameters

num_classes = 3
num_features = X_train_tfidf.shape[1]

# weights matrix: each row is weights for one class
np.random.seed(42)
W = np.random.randn(num_classes, num_features) * 0.01
b = np.zeros(num_classes)

print(f"W shape: {W.shape}")
print(f"b shape: {b.shape}")

W shape: (3, 2959)
b shape: (3,)


In [15]:
# convert labels to one-hot encoding

def one_hot_encode(y, num_classes):
    y_one_hot = np.zeros((len(y), num_classes))
    y_one_hot[np.arange(len(y)), y] = 1
    return y_one_hot

y_train_onehot = one_hot_encode(y_train.values, num_classes)
y_train_onehot.shape

(1850, 3)

In [16]:
# training using gradient descent
# loss: cross-entropy = -sum(y_true * log(y_pred))

learning_rate = 1.0
num_epochs = 500

for epoch in range(num_epochs):
    # forward pass: compute predictions
    logits = X_train_tfidf @ W.T + b
    y_pred = softmax(logits)
    
    # compute loss (cross-entropy)
    loss = -np.mean(np.sum(y_train_onehot * np.log(y_pred + 1e-10), axis=1))
    
    # backward pass: compute gradients
    error = y_pred - y_train_onehot
    
    dW = (error.T @ X_train_tfidf) / len(X_train_tfidf)
    db = np.mean(error, axis=0)
    
    # update parameters
    W -= learning_rate * dW
    b -= learning_rate * db
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {loss:.4f}")

Epoch 100/500, Loss: 0.9599
Epoch 200/500, Loss: 0.8890
Epoch 300/500, Loss: 0.8331
Epoch 400/500, Loss: 0.7873
Epoch 500/500, Loss: 0.7486


In [17]:
# make predictions on test set

logits_test = X_test_tfidf @ W.T + b
y_pred_probs = softmax(logits_test)
y_pred = np.argmax(y_pred_probs, axis=1)

y_pred[:10]

array([0, 2, 2, 0, 2, 2, 0, 2, 2, 2], dtype=int64)

In [18]:
# compute accuracy

accuracy = np.mean(y_pred == y_test.values)
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 64.79%


In [19]:
# compute precision, recall, f1-score for each class

classes = [0, 1, 2]
class_names = ['negative', 'neutral', 'positive']

print("\nClassification Report:")
print(f"{'Class':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
print("-" * 50)

for i, class_name in zip(classes, class_names):
    # True Positives, False Positives, False Negatives
    tp = np.sum((y_pred == i) & (y_test.values == i))
    fp = np.sum((y_pred == i) & (y_test.values != i))
    fn = np.sum((y_pred != i) & (y_test.values == i))
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"{class_name:<12} {precision:<12.2f} {recall:<12.2f} {f1:<12.2f}")


Classification Report:
Class        Precision    Recall       F1-Score    
--------------------------------------------------
negative     0.60         0.76         0.67        
neutral      0.91         0.11         0.19        
positive     0.69         0.80         0.74        


In [20]:
# confusion matrix

confusion_mat = np.zeros((num_classes, num_classes), dtype=int)

for true_label, pred_label in zip(y_test.values, y_pred):
    confusion_mat[true_label][pred_label] += 1

print("\nConfusion Matrix:")
print(confusion_mat)


Confusion Matrix:
[[131   1  41]
 [ 50  10  32]
 [ 39   0 159]]


In [21]:
# show some predictions

label_decode = {
    0: 'negative',
    1: 'neutral',
    2: 'positive'
}

results_df = pd.DataFrame({
    'aspect_context': X_test.values[:10],
    'true_label': [label_decode[label] for label in y_test.values[:10]],
    'predicted_label': [label_decode[label] for label in y_pred[:10]]
})

results_df

,aspect_context,true_label,predicted_label
0,netbook was desinged for was [ASPECT] and boy ...,positive,negative
1,"either way, the [ASPECT] is unsatisfactory.",negative,positive
2,"great os, fabulous improvements to the existin...",positive,positive
3,didn't [ASPECT] when shipped from walmart.com but,negative,negative
4,the [ASPECT] is like no other laptop,neutral,positive
5,the [ASPECT] is incredible and i am,positive,positive
6,"all dark, power light steady, [ASPECT] steady ...",negative,negative
7,"to do simple things, like [ASPECT]",negative,positive
8,the difference is it's a whole lot of fun usin...,positive,positive
9,"screen, keyboard, and mouse: if you cant see y...",neutral,positive
